# HIT-UAV × YOLO26 红外无人机弱小目标检测 — Colab Pro 总控启动台

**使用前 2 步检查（避免白烧 Compute Units）**：
1. 右上角「更改运行时类型」→ 选 **L4 GPU**（约 1.94 CU/h，22.5GB 显存 + 53GB 内存）。
2. 左侧钥匙图标 **Secrets（密钥）** → 添加密钥 `GITHUB_TOKEN`（你的 GitHub Personal Access Token，勾选允许此笔记本访问），把下方 `GITHUB_REPO` 改成你的私有仓库地址。

**推荐执行顺序**：
- **第 1 次运行（Phase 0 源码体检 + 数据集准备，约 2 分钟 / <0.05 CU）**：保持 `RUN_PHASE0_INSPECT = True`、`RUN_PREPARE_DATA = True`、`RUN_TRAINING = False`，跑完后把 Drive 里的 `audit/phase0_yolo26_source_audit.md` 发给哈基米核对源码。
- **第 2 次运行（烟雾测试，约 2 分钟 / <0.06 CU）**：`RUN_PHASE0_INSPECT = False`、`RUN_TRAINING = True`、`SMOKE_TEST = True`、`EXP_LIST = ["p0_yolo26n_musgd"]`，确认全链路无报错。
- **第 3 次运行（Phase 0 正式基线摸底，约 1.5 小时 / ~3 CU）**：`SMOKE_TEST = False`、`EXP_LIST = ["p0_yolo26n_musgd", "p0_yolo26n_sgd", "p0_yolo11n", "p0_yolov8n"]`、`AUTO_UNASSIGN = True`（跑完自动断连省 CU！）。

In [ ]:
# =============================================================================
# 🎛️ 总控开关区（平时改实验只需要改这里！）
# =============================================================================

# 1. 你的 GitHub 仓库（格式: "用户名/仓库名"，如 "hxttty/ir-yolo26-paper"）
GITHUB_REPO = "hxttty/ir-yolo26-paper"
GITHUB_BRANCH = "main"
USE_PRIVATE_TOKEN = True   # 私有仓库设为 True（从 Colab Secrets 读取 GITHUB_TOKEN）

# 2. Google Drive 持久存储根目录（权重、日志、数据集缓存、结果总表都在这里）
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/ir-yolo26-paper"

# 3. 如果 HIT-UAV 官方 GitHub 不含原始图片，把下载好的 raw zip 放进 Drive 填路径；否则留 None 自动拉取
RAW_HITUAV_PATH = None  # 例: "/content/drive/MyDrive/ir-yolo26-paper/raw/hit-uav.zip"

# 4. 任务总开关
RUN_PHASE0_INSPECT = True    # 第一次必跑：打印并保存实装 YOLO26 源码体检报告
RUN_PREPARE_DATA   = True    # 准备/秒解压 HIT-UAV（首次建缓存包，之后 3 秒直出）
RUN_TRAINING       = False   # 确认源码和数据集无误后改成 True 开训

# 5. 实验队列与安全开关（支持一次排队跑多个预设！）
EXP_LIST = [
    "p0_yolo26n_musgd",
    # "p0_yolo26n_sgd",
    # "p0_yolo11n",
    # "p0_yolov8n",
]
SMOKE_TEST    = True   # True=仅跑 2 epoch 验证全流程（强烈建议每次改完代码先 True 测 1 分钟！）
SEED          = 42     # 3 种子验证时依次改 0 / 42 / 3407
RESUME        = True   # True=若上次中途断连，自动从 Drive 的 last.pt 断点续训
EXPORT_ONNX   = False  # True=训完顺便导出 ONNX 并测 CPU 端单图延迟（正式出表时打开）
AUTO_UNASSIGN = False  # 跑长时间正式训练时务必改 True：跑完/报错后自动销毁运行时停表省 CU！


In [ ]:
# =============================================================================
# Cell 2: 挂载 Google Drive + 检查 GPU 硬件
# =============================================================================
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
Path(DRIVE_PROJECT_DIR).mkdir(parents=True, exist_ok=True)
!nvidia-smi


In [ ]:
# =============================================================================
# Cell 3: 从 GitHub 拉取/更新最新代码（Token 仅走内存 Header，不落盘）+ 安装依赖
# =============================================================================
import base64
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/ir-yolo26-paper")

if GITHUB_REPO.startswith("YOUR_GITHUB_USERNAME"):
    raise ValueError("请先在 Cell 1 把 GITHUB_REPO 改成你自己的 GitHub 仓库地址！")

clean_url = f"https://github.com/{GITHUB_REPO}.git"
git_auth_args = []
if USE_PRIVATE_TOKEN:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if not token:
        raise ValueError("未读取到 Colab Secret `GITHUB_TOKEN`，请在左侧钥匙图标里添加并授权。")
    auth_b64 = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_auth_args = ["-c", f"http.extraheader=AUTHORIZATION: basic {auth_b64}"]

try:
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), *git_auth_args, "fetch", "origin", GITHUB_BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{GITHUB_BRANCH}"], check=True)
    else:
        subprocess.run(["git", *git_auth_args, "clone", "-b", GITHUB_BRANCH, clean_url, str(REPO_DIR)], check=True)
except subprocess.CalledProcessError as exc:
    raise RuntimeError(
        f"Git 拉取失败 (exit={exc.returncode})。请检查 GITHUB_REPO 拼写及 GITHUB_TOKEN 对该私有仓库的 Contents 只读权限。"
    ) from None

%cd /content/ir-yolo26-paper
!git log -1 --oneline
!pip install -q -r requirements.txt


In [ ]:
# =============================================================================
# Cell 4: [Phase 0 必做] 打印并保存实装 Ultralytics YOLO26 源码体检报告
# =============================================================================
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/ir-yolo26-paper")

if RUN_PHASE0_INSPECT:
    audit_out = Path(DRIVE_PROJECT_DIR) / "audit" / "phase0_yolo26_source_audit.md"
    subprocess.run(
        ["python", "-m", "src.inspect_yolo26", "--out-md", str(audit_out)],
        cwd=str(REPO_DIR),
        check=True,
    )
    print(f"\n✅ 体检报告已保存到 Drive: {audit_out}")
    print("👉 请下载或复制这份 md 文件发给哈基米，确认精确源码后再写改动 A/B/C1/C2 补丁！")
else:
    print("[SKIP] RUN_PHASE0_INSPECT=False")


In [ ]:
# =============================================================================
# Cell 5: 准备 / 秒解压 HIT-UAV 数据集到 Colab 本地高速盘 (/content/datasets)
# =============================================================================
import subprocess
from pathlib import Path

REPO_DIR     = Path("/content/ir-yolo26-paper")
DATASET_ROOT = Path("/content/datasets")
DRIVE_BUNDLE = Path(DRIVE_PROJECT_DIR) / "datasets" / "hit_uav_bundle.zip"
VIS_DIR      = Path(DRIVE_PROJECT_DIR) / "audit" / "sanity_vis"

if RUN_PREPARE_DATA:
    cmd = [
        "python", "-m", "src.prepare_hituav",
        "--out-root", str(DATASET_ROOT),
        "--drive-bundle", str(DRIVE_BUNDLE),
        "--vis-dir", str(VIS_DIR),
    ]
    if RAW_HITUAV_PATH:
        cmd += ["--raw-dir", str(RAW_HITUAV_PATH)]
    subprocess.run(cmd, cwd=str(REPO_DIR), check=True)
else:
    print("[SKIP] RUN_PREPARE_DATA=False")


In [ ]:
# =============================================================================
# Cell 6: 执行实验队列 + 自动 Drive 存档 + try...finally 自动断连停表防烧 CU
# =============================================================================
import subprocess
import traceback
from pathlib import Path
from google.colab import runtime

REPO_DIR = Path("/content/ir-yolo26-paper")

if RUN_TRAINING:
    try:
        for exp_name in EXP_LIST:
            print("\n" + "#" * 72)
            print(f"🚀 开始运行实验: {exp_name} (smoke_test={SMOKE_TEST}, seed={SEED})")
            print("#" * 72)
            cmd = [
                "python", "-m", "src.train_runner",
                "--exp", exp_name,
                "--dataset-root", "/content/datasets",
                "--work-dir", "/content/work",
                "--drive-dir", DRIVE_PROJECT_DIR,
                "--seed", str(SEED),
            ]
            if SMOKE_TEST:
                cmd.append("--smoke-test")
            if not RESUME:
                cmd.append("--no-resume")
            if EXPORT_ONNX:
                cmd.append("--export-onnx")
            subprocess.run(cmd, cwd=str(REPO_DIR), check=True)
        print("\n🎉 队列内所有实验运行并归档完毕！")
    except Exception:
        traceback.print_exc()
        raise
    finally:
        if AUTO_UNASSIGN:
            print("\n🛑 AUTO_UNASSIGN=True -> 正在自动释放 Colab GPU 运行时，停止扣费！")
            runtime.unassign()
else:
    print("[SKIP] RUN_TRAINING=False（如需查看全部预设名称，可运行 `!python -m src.train_runner --list`）")
